# Líneas base — NeuroVoz

Entrena las 11 arquitecturas de línea base sobre NeuroVoz (misma malla y esquema
que PC-GITA: `StratifiedKFold` de 5 particiones sobre las formas de onda crudas)
y guarda el resultado en `results/baseline_results_neurovoz.json`, análogo al de
PC-GITA que consume `09_resultados_unificado.ipynb`.


## 0. Configuración

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = "/Users/napster/Documents/upm"
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import numpy as np, torch

TARGET_SR    = 16_000
TASK         = "PATAKA"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

NEUROVOZ_ROOT = "data/neurovoz"
AUDIO_DIR = f"{NEUROVOZ_ROOT}/audios"
META_HC   = f"{NEUROVOZ_ROOT}/metadata/metadata_hc.csv"
META_PD   = f"{NEUROVOZ_ROOT}/metadata/metadata_pd.csv"

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print("device =", DEVICE)

device = mps


## 1. Carga y preprocesado de NeuroVoz (formas de onda crudas)

In [2]:
import pandas as pd, soundfile as sf
from math import gcd
from scipy.signal import resample_poly

def load_neurovoz_metadata(meta_hc_path, meta_pd_path, task=TASK):
    hc = pd.read_csv(meta_hc_path); pd_ = pd.read_csv(meta_pd_path)
    df = pd.concat([hc, pd_], ignore_index=True)
    df["filename"] = df["Audio"].str.split("/").str[-1]
    df["task"] = df["filename"].str.extract(r"^[A-Z]+_(.+)_\d+\.wav")
    df = df[df["task"] == task].copy()
    df["label"] = (df["Group"] == "PD").astype(int)
    df["audio_path"] = df["filename"].apply(lambda f: os.path.join(AUDIO_DIR, f))
    return df[["label", "audio_path"]].reset_index(drop=True)

def load_and_resample(path, target_sr=TARGET_SR):
    wav, sr = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim > 1: wav = wav.mean(axis=1)
    if sr != target_sr:
        g = gcd(sr, target_sr); wav = resample_poly(wav, target_sr // g, sr // g)
    return wav.astype(np.float32)

def preprocess_waveform(x, pre_emphasis=0.97):
    x = x - np.mean(x)
    x = np.append(x[0], x[1:] - pre_emphasis * x[:-1])
    peak = np.max(np.abs(x))
    return x / peak if peak > 0 else x

df = load_neurovoz_metadata(META_HC, META_PD, TASK)
waveforms_proc = [preprocess_waveform(load_and_resample(p)) for p in df["audio_path"]]
labels = df["label"].to_numpy()
print("Sujetos:", len(waveforms_proc), "| clases:",
      dict(zip(*np.unique(labels, return_counts=True))))

Sujetos: 99 | clases: {np.int64(0): np.int64(50), np.int64(1): np.int64(49)}


## 2. Entrenamiento de las 11 líneas base (cacheado)

In [3]:
import json
from pathlib import Path
from src.models import (CNN1D, Cnn1dLight, SincNet, WaveNetLike, TemporalTransformer,
                        BiLSTM_CNN, CNNAttention, ECAPA_TDNN, Res1D)
from src.models_marta import InceptionTime, CDIL_CNN
from src.training import run_kfold_search

# ~4 h desde cero. Se cachea en disco:
#   - si existe el JSON -> se carga (segundos)
#   - si no -> se entrena una vez y se guarda
# Borra el fichero para forzar un reentrenamiento completo.
BASELINE_JSON = Path("results/baseline_results_neurovoz.json")

if BASELINE_JSON.exists():
    baseline_results = json.load(open(BASELINE_JSON))
    print("cargado desde", BASELINE_JSON, "->", list(baseline_results))
else:
    WINDOW_LEN, HOP_LEN = 12000, 6000
    lite = {"lr":[5e-4, 1e-4], "batch_size":[4, 8]}
    HYPERPARAM_GRID = {k: lite for k in
        ["CNN1D","CNN1D-Light","SincNet","WaveNet-like","TemporalTransformer",
         "BiLSTM-CNN","CNN-Attention","ECAPA-TDNN","Res1D","InceptionTime","CDIL-CNN"]}
    MODEL_CONFIG = {
        "CNN1D":{"class":CNN1D,"patience":10,"epochs":50},
        "CNN1D-Light":{"class":Cnn1dLight,"patience":10,"epochs":50},
        "SincNet":{"class":SincNet,"patience":10,"epochs":50},
        "WaveNet-like":{"class":WaveNetLike,"patience":10,"epochs":50},
        "TemporalTransformer":{"class":TemporalTransformer,"patience":10,"epochs":50},
        "BiLSTM-CNN":{"class":BiLSTM_CNN,"patience":10,"epochs":50},
        "CNN-Attention":{"class":CNNAttention,"patience":10,"epochs":50},
        "ECAPA-TDNN":{"class":ECAPA_TDNN,"patience":10,"epochs":50},
        "Res1D":{"class":Res1D,"patience":10,"epochs":50},
        "InceptionTime":{"class":InceptionTime,"patience":10,"epochs":50},
        "CDIL-CNN":{"class":CDIL_CNN,"patience":10,"epochs":50},
    }
    baseline_results = run_kfold_search(df=df, waveforms_processed=waveforms_proc,
        model_config=MODEL_CONFIG, hyperparam_grid=HYPERPARAM_GRID,
        n_folds=5, random_state=RANDOM_STATE, window_len=WINDOW_LEN, hop_len=HOP_LEN)
    json.dump(
        {k: {kk: vv for kk, vv in v.items() if kk != "fold_details"}
         for k, v in baseline_results.items()},
        open(BASELINE_JSON, "w"), indent=2, default=float)
    print("entrenado y guardado en", BASELINE_JSON, "->", list(baseline_results))

2026-07-12 23:32:12,179 [INFO] src.config: 
2026-07-12 23:32:12,179 [INFO] src.config:   CNN1D
2026-07-12 23:32:12,180 [INFO] src.config: ======================================================================


KeyboardInterrupt: 

## 3. Tabla resumen (AUC y Accuracy por modelo)

In [ ]:
order = sorted(baseline_results, key=lambda m: baseline_results[m]["auc_mean"], reverse=True)
print(f'{"Modelo":22} {"AUC":^17} {"Accuracy":^17}')
for m in order:
    r = baseline_results[m]
    print(f'{m:22} {r["auc_mean"]:.3f} ± {r["auc_std"]:.3f}   '
          f'{r["acc_mean"]:.3f} ± {r["acc_std"]:.3f}')

## 4. Figura: sensibilidad vs especificidad

In [ ]:
import matplotlib.pyplot as plt

IMAGES_DIR = os.path.join(PROJECT_ROOT, "images"); os.makedirs(IMAGES_DIR, exist_ok=True)

d = baseline_results
order = sorted(d, key=lambda m: d[m]["auc_mean"])
sens = [d[m]["sens_mean"] for m in order]
spec = [d[m]["spec_mean"] for m in order]
x = np.arange(len(order)); w = 0.4
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(x - w / 2, sens, w, label="Sensibilidad", color="#2A9D8F")
ax.bar(x + w / 2, spec, w, label="Especificidad", color="#8E7CC3")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=35, ha="right")
ax.set_ylim(0, 1.05); ax.legend()
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
out_png = os.path.join(IMAGES_DIR, "res_baselines_sens_spec_nv.png")
plt.savefig(out_png, dpi=200, facecolor="white", bbox_inches="tight")
plt.show()
print("guardada", out_png)